23.09.2026, 8AM Tutorial
# LLM, Transformers Lab

2 major blocks: encoder layer and decoder layers.

Encoder: input sequence + self-attention + output sequence routed to decoder
Decoder:

GPT-3 used only the decoder architecture. Now both are used.
Decoder -> Next word prediction
[Latent space representation?]

Encoder: representational learning, such as sentiment analysis

T5 - translation model
ViT, GPT: Decoder only

Encoder-only - Bert



Task 1: encoder + decoder

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

# We are swapping out the untrained bert2bert for a fully trained
# Encoder-Decoder translation model (English to Spanish)
model_name = "Helsinki-NLP/opus-mt-en-es"

print(f"Loading tokenizer and model ({model_name})...")
# MarianTokenizer and MarianMTModel are optimized specifically for translation
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Your input text
source_text = "plants create energy through a process known as photosynthesis"

# Tokenize the input text
inputs = tokenizer(source_text, return_tensors="pt")

print("Generating translation...")
# Generate the sequence
outputs = model.generate(
    **inputs,
    max_length=50,
    num_beams=4,
    early_stopping=True
)

# Decode the generated tokens back into a readable string
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\nTarget Language: Spanish")
print(f"Input (English): {source_text}")
print(f"Output (Spanish): {generated_text}")

Loading tokenizer and model (Helsinki-NLP/opus-mt-en-es)...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Generating translation...

Target Language: Spanish
Input (English): plants create energy through a process known as photosynthesis
Output (Spanish): las plantas crean energía a través de un proceso conocido como fotosíntesis


Task 2: Next token generation (decoder only)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load a Causal Language Model designed for next-token prediction
model_name = "gpt2"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Define the input sentence (Context)
input_text = "AI is an abbreviation for"
inputs = tokenizer(input_text, return_tensors="pt")

print(f"\nInput context: '{input_text}'\n")

# =====================================================================
# Method A: Literal Next Token Prediction (Looking at Logits)
# =====================================================================
# Pass the input through the model to get the raw scores (logits) for every word in the vocabulary
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# We only care about the predictions for the very last token in our sequence
next_token_logits = logits[0, -1, :]

# Get the token ID with the highest probability score
predicted_token_id = torch.argmax(next_token_logits).item()

# Decode that ID back into a human-readable word
predicted_word = tokenizer.decode(predicted_token_id)
print(f"Method A - The single most likely next token is: '{predicted_word}'")

# =====================================================================
# Method B: Auto-completing the sentence (Generating multiple tokens)
# =====================================================================
# The .generate() function automatically feeds the predicted token back into the model
# in a loop to generate a sequence of words.
generation_output = model.generate(
    inputs.input_ids,
    max_new_tokens=5, # Predict the next 5 tokens
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False   # Greedy decoding (always pick the highest probability)
)

completed_sentence = tokenizer.decode(generation_output[0], skip_special_tokens=True)
print(f"Method B - Generating the next 5 tokens: '{completed_sentence}'")

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Input context: 'AI is an abbreviation for'

Method A - The single most likely next token is: ' the'
Method B - Generating the next 5 tokens: 'AI is an abbreviation for the International Monetary Fund.'


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load a Causal Language Model designed for next-token prediction
model_name = "gpt2"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Define the input sentence (Context)
input_text = "The future of AI is"
inputs = tokenizer(input_text, return_tensors="pt")

print(f"\nInput context: '{input_text}'\n")

# =====================================================================
# Method A: Literal Next Token Prediction (Looking at Logits)
# =====================================================================
# Pass the input through the model to get the raw scores (logits) for every word in the vocabulary
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# We only care about the predictions for the very last token in our sequence
next_token_logits = logits[0, -1, :]

# Get the token ID with the highest probability score
predicted_token_id = torch.argmax(next_token_logits).item()

# Decode that ID back into a human-readable word
predicted_word = tokenizer.decode(predicted_token_id)
print(f"Method A - The single most likely next token is: '{predicted_word}'")

# =====================================================================
# Method B: Auto-completing the sentence (Generating multiple tokens)
# =====================================================================
# The .generate() function automatically feeds the predicted token back into the model
# in a loop to generate a sequence of words.
generation_output = model.generate(
    inputs.input_ids,
    max_new_tokens=25, # Predict the next 5 tokens
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True   # Greedy decoding (always pick the highest probability)
)

completed_sentence = tokenizer.decode(generation_output[0], skip_special_tokens=True)
print(f"Method B - Generating the next 5 tokens: '{completed_sentence}'")

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Input context: 'The future of AI is'

Method A - The single most likely next token is: ' uncertain'
Method B - Generating the next 5 tokens: 'The future of AI is open door for the computer program to be able to do intelligent driving in the future. We have already discovered that it is possible'


Task 3 - Sentiment Analysis using Encoder-only architecture

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Load an Encoder-only model fine-tuned for Sentiment Analysis
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 2. Define input sentences (one positive, one negative)
sentences = [
    "I am absolutely amazed by how efficiently plants create energy!",
    "The recent drought has severely damaged the crops, which is very disappointing.",
    "I am happy.",
    "I am upset.",
    "Mitochondria is the powerhouse of the cell.",
    "ifweihfweoinoidjwqp",
    "This film was made in India.",
    "This film was made in Afghanistan.",
    "Constructive algorithms",
    "Destructive algorithms"
]

print("\n--- Sentiment Analysis Results ---")

# 3. Process each sentence
for text in sentences:
    # Tokenize the input text into integer IDs
    inputs = tokenizer(text, return_tensors="pt")

    # Pass the tokens through the encoder-only model
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Raw scores for [Negative, Positive]

    # Find the index with the highest score
    predicted_class_id = torch.argmax(logits, dim=-1).item()

    # Convert raw logits to probabilities using Softmax
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    confidence = probabilities[0][predicted_class_id].item() * 100

    # Map the class ID back to the human-readable label (0 -> NEGATIVE, 1 -> POSITIVE)
    label = model.config.id2label[predicted_class_id]

    print(f"\nSentence: '{text}'")
    print(f"Sentiment: {label} (Confidence: {confidence:.2f}%)")

Loading distilbert-base-uncased-finetuned-sst-2-english...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- Sentiment Analysis Results ---

Sentence: 'I am absolutely amazed by how efficiently plants create energy!'
Sentiment: POSITIVE (Confidence: 99.98%)

Sentence: 'The recent drought has severely damaged the crops, which is very disappointing.'
Sentiment: NEGATIVE (Confidence: 99.97%)

Sentence: 'I am happy.'
Sentiment: POSITIVE (Confidence: 99.99%)

Sentence: 'I am upset.'
Sentiment: NEGATIVE (Confidence: 99.90%)

Sentence: 'Mitochondria is the powerhouse of the cell.'
Sentiment: POSITIVE (Confidence: 97.78%)

Sentence: 'ifweihfweoinoidjwqp'
Sentiment: NEGATIVE (Confidence: 98.36%)

Sentence: 'This film was made in India.'
Sentiment: POSITIVE (Confidence: 99.82%)

Sentence: 'This film was made in Afghanistan.'
Sentiment: NEGATIVE (Confidence: 88.77%)

Sentence: 'Constructive algorithms'
Sentiment: POSITIVE (Confidence: 99.97%)

Sentence: 'Destructive algorithms'
Sentiment: NEGATIVE (Confidence: 99.97%)


# Running Ollama


In [ ]:
!pip install colab-xterm
%load-ext colabxterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 5.2 MB/s eta 0:00:00


UsageError: Line magic function `%load-ext` not found.
